In [0]:
import time
 
def timed(sql: str, label: str):
    t0 = time.time()
    n = spark.sql(sql).count()
    dt = time.time() - t0
    print(f"{label}: {dt:.2f}s  ({n:,} rows)")
    return dt
 
query = """
SELECT z.borough, COUNT(*) AS trips, AVG(f.total_amount) AS avg_total
FROM nyc_taxi.gold.fact_trips f
JOIN nyc_taxi.gold.dim_taxi_zone z ON f.pickup_zone_key = z.zone_key
WHERE f.date_key BETWEEN "2024-03-01" AND "2024-03-31"
GROUP BY z.borough
"""
 
before = timed(query, "BEFORE")

In [0]:
spark.sql("EXPLAIN " + query).show(truncate=False)

In [0]:
%sql
OPTIMIZE nyc_taxi.gold.fact_trips;
DESCRIBE HISTORY nyc_taxi.gold.fact_trips;

In [0]:
%sql
OPTIMIZE nyc_taxi.gold.fact_trips ZORDER BY (pickup_zone_key);
 
-- Or the newer, self-tuning alternative:
-- ALTER TABLE nyc_taxi.gold.fact_trips CLUSTER BY (pickup_zone_key);

In [0]:
from pyspark.sql.functions import broadcast
 
fact = spark.table("nyc_taxi.gold.fact_trips")
zone = spark.table("nyc_taxi.gold.dim_taxi_zone")
 
result = fact.join(broadcast(zone), fact.pickup_zone_key == zone.zone_key)

In [0]:
%sql
ANALYZE TABLE nyc_taxi.gold.fact_trips COMPUTE STATISTICS FOR ALL COLUMNS;

In [0]:
after = timed(query, "AFTER")
print(f"improvement: {(1 - after / before) * 100:.1f}%")

In [0]:
%sql
CREATE OR REPLACE FUNCTION nyc_taxi.gold.borough_filter(zone_key INT)
RETURN
  is_account_group_member("taxi_admins")
  OR zone_key IN (
    SELECT zone_key FROM nyc_taxi.gold.dim_taxi_zone WHERE borough = "Manhattan"
  );
 
ALTER TABLE nyc_taxi.gold.fact_trips
  SET ROW FILTER nyc_taxi.gold.borough_filter ON (pickup_zone_key);

In [0]:
%sql
CREATE OR REPLACE FUNCTION nyc_taxi.gold.mask_amount(amount DOUBLE)
RETURN CASE
  WHEN is_account_group_member("taxi_admins") THEN amount
  ELSE NULL
END;
 
ALTER TABLE nyc_taxi.gold.fact_trips
  ALTER COLUMN total_amount SET MASK nyc_taxi.gold.mask_amount;

In [0]:
timed(query, "WARM_BASELINE")

In [0]:
print(query)

In [0]:
spark.sql(query).show()

In [0]:
%sql
SELECT * FROM nyc_taxi.gold.dim_taxi_zone LIMIT 20;

In [0]:
%sql
DESCRIBE HISTORY nyc_taxi.gold.dim_taxi_zone;

In [0]:
%sql
SELECT f.pickup_zone_key, z.zone_key, z.borough
FROM nyc_taxi.gold.fact_trips f
JOIN nyc_taxi.gold.dim_taxi_zone z ON f.pickup_zone_key = z.zone_key
WHERE f.date_key BETWEEN '2024-03-01' AND '2024-03-31'
LIMIT 20;

In [0]:
spark.sql(query).show()

In [0]:
%sql
SELECT COUNT(*) FROM nyc_taxi.bronze.yellow_tripdata WHERE _source_year = 2024 AND _source_month = 2;

In [0]:
%sql
SELECT COUNT(*) FROM nyc_taxi.gold.fact_trips WHERE date_key BETWEEN '2024-02-01' AND '2024-02-28';

In [0]:
%sql
SELECT COUNT(*) FROM nyc_taxi.bronze.yellow_tripdata WHERE _source_year = 2024 AND _source_month = 1;

In [0]:
%sql
SELECT COUNT(*) FROM nyc_taxi.gold.fact_trips WHERE date_key BETWEEN '2024-01-01' AND '2024-01-31';